# FinanceBench Current Retrieval + Hybrid Search Evaluation

Notebook nay danh gia kha nang truy xuat hien tai tren retrieval benchmark da tao tu FinanceBench.

Input:
- `outputs/financebench_eval/chunks.jsonl`
- `outputs/financebench_eval/qrels/queries.jsonl`
- `outputs/financebench_eval/qrels/qrels.jsonl`

Phuong phap:
- Dense retrieval: BGE embedding + cosine / inner product.
- Sparse retrieval: TF-IDF lexical search.
- Hybrid retrieval: ket hop dense score va sparse score.

Metrics:
- Precision@k
- Recall@k
- Hit@k
- MRR@k

In [1]:
from pathlib import Path
import json
import math
import sys
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

ROOT = Path('..').resolve()
sys.path.append(str(ROOT / 'src'))

from common.bge_embedder import DEFAULT_BGE_MODEL, load_bge_model

CHUNKS_PATH = ROOT / 'outputs/financebench_eval/chunks.jsonl'
QUERIES_PATH = ROOT / 'outputs/financebench_eval/qrels/queries.jsonl'
QRELS_PATH = ROOT / 'outputs/financebench_eval/qrels/qrels.jsonl'
OUT_DIR = ROOT / 'outputs/financebench_eval/hybrid_retrieval_eval'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = DEFAULT_BGE_MODEL
BATCH_SIZE = 32
DEVICE = None  # e.g. 'cuda' or 'cpu'
K_VALUES = [1, 3, 5, 10]
TOP_K = max(K_VALUES)
HYBRID_ALPHA = 0.60  # dense weight; sparse weight = 1 - alpha

pd.set_option('display.max_colwidth', 160)

c:\Users\pntha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_jsonl(path: Path) -> list[dict]:
    if not path.exists() or path.stat().st_size == 0:
        return []
    with path.open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

def chunk_text(chunk: dict) -> str:
    return str(chunk.get('text') or chunk.get('summary') or '')

def chunk_id(chunk: dict) -> str:
    return str(chunk.get('chunk_id') or chunk.get('id'))

def page_range(chunk: dict):
    start = chunk.get('page_start', chunk.get('page'))
    end = chunk.get('page_end', chunk.get('page'))
    try:
        start = int(start) if start is not None else None
        end = int(end) if end is not None else start
    except (TypeError, ValueError):
        return None, None
    return start, end

def minmax_row(scores: np.ndarray) -> np.ndarray:
    lo = scores.min(axis=1, keepdims=True)
    hi = scores.max(axis=1, keepdims=True)
    denom = np.where((hi - lo) == 0, 1.0, hi - lo)
    return (scores - lo) / denom

def topk_from_scores(score_matrix: np.ndarray, ids: list[str], queries: list[dict], chunks_by_id: dict[str, dict], top_k: int, method: str) -> list[dict]:
    rows = []
    for qi, query in enumerate(queries):
        scores = score_matrix[qi]
        if top_k < len(scores):
            candidate_idx = np.argpartition(-scores, top_k - 1)[:top_k]
            candidate_idx = candidate_idx[np.argsort(-scores[candidate_idx])]
        else:
            candidate_idx = np.argsort(-scores)
        for rank, idx in enumerate(candidate_idx[:top_k], start=1):
            cid = ids[int(idx)]
            chunk = chunks_by_id[cid]
            page_start, page_end = page_range(chunk)
            rows.append({
                'method': method,
                'query_id': query['query_id'],
                'rank': rank,
                'chunk_id': cid,
                'score': float(scores[int(idx)]),
                'doc_name': chunk.get('doc_name'),
                'page_start': page_start,
                'page_end': page_end,
                'chunk_type': chunk.get('chunk_type') or chunk.get('modality'),
                'text_preview': chunk_text(chunk)[:300],
            })
    return rows

## 1. Load benchmark retrieval

In [3]:
chunks = load_jsonl(CHUNKS_PATH)
queries = load_jsonl(QUERIES_PATH)
qrels = load_jsonl(QRELS_PATH)

ids = [chunk_id(c) for c in chunks]
texts = [chunk_text(c) for c in chunks]
query_texts = [q['question'] for q in queries]
chunks_by_id = {chunk_id(c): c for c in chunks}

qrels_by_query = defaultdict(set)
for row in qrels:
    qrels_by_query[row['query_id']].add(row['chunk_id'])

print('chunks:', len(chunks))
print('queries:', len(queries))
print('qrels:', len(qrels))
print('queries with qrels:', len(qrels_by_query))

chunks: 11948
queries: 150
qrels: 242
queries with qrels: 150


## 2. Dense retrieval: BGE embedding

In [4]:
chunk_emb_path = OUT_DIR / 'chunk_embeddings.npy'
query_emb_path = OUT_DIR / 'query_embeddings.npy'
ids_path = OUT_DIR / 'chunk_ids.json'
query_ids_path = OUT_DIR / 'query_ids.json'

need_embed = True
if chunk_emb_path.exists() and query_emb_path.exists() and ids_path.exists() and query_ids_path.exists():
    cached_ids = json.loads(ids_path.read_text(encoding='utf-8'))
    cached_query_ids = json.loads(query_ids_path.read_text(encoding='utf-8'))
    if cached_ids == ids and cached_query_ids == [q['query_id'] for q in queries]:
        need_embed = False

if need_embed:
    embedder = load_bge_model(model_name=MODEL_NAME, batch_size=BATCH_SIZE, device=DEVICE)
    chunk_embeddings = embedder.encode_documents(texts)
    query_embeddings = embedder.encode_queries(query_texts)
    np.save(chunk_emb_path, chunk_embeddings)
    np.save(query_emb_path, query_embeddings)
    ids_path.write_text(json.dumps(ids, ensure_ascii=False, indent=2), encoding='utf-8')
    query_ids_path.write_text(json.dumps([q['query_id'] for q in queries], ensure_ascii=False, indent=2), encoding='utf-8')
else:
    chunk_embeddings = np.load(chunk_emb_path)
    query_embeddings = np.load(query_emb_path)

chunk_embeddings.shape, query_embeddings.shape

Batches:   3%|▎         | 13/374 [08:04<3:44:12, 37.27s/it]


KeyboardInterrupt: 

In [ ]:
dense_scores = query_embeddings @ chunk_embeddings.T
dense_results = topk_from_scores(dense_scores, ids, queries, chunks_by_id, TOP_K, method='dense_bge')
write_jsonl(OUT_DIR / 'dense_retrieval_results.jsonl', dense_results)
dense_scores.shape, len(dense_results)

## 3. Sparse retrieval: TF-IDF lexical search

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
)
chunk_tfidf = vectorizer.fit_transform(texts)
query_tfidf = vectorizer.transform(query_texts)
chunk_tfidf = normalize(chunk_tfidf, norm='l2', axis=1)
query_tfidf = normalize(query_tfidf, norm='l2', axis=1)
sparse_scores = (query_tfidf @ chunk_tfidf.T).toarray().astype('float32')
sparse_results = topk_from_scores(sparse_scores, ids, queries, chunks_by_id, TOP_K, method='sparse_tfidf')
write_jsonl(OUT_DIR / 'sparse_retrieval_results.jsonl', sparse_results)
chunk_tfidf.shape, sparse_scores.shape, len(sparse_results)

## 4. Hybrid retrieval

In [ ]:
dense_norm = minmax_row(dense_scores.astype('float32'))
sparse_norm = minmax_row(sparse_scores.astype('float32'))
hybrid_scores = HYBRID_ALPHA * dense_norm + (1.0 - HYBRID_ALPHA) * sparse_norm
hybrid_results = topk_from_scores(hybrid_scores, ids, queries, chunks_by_id, TOP_K, method=f'hybrid_bge_tfidf_alpha_{HYBRID_ALPHA:.2f}')
write_jsonl(OUT_DIR / 'hybrid_retrieval_results.jsonl', hybrid_results)
hybrid_scores.shape, len(hybrid_results)

## 5. Metrics

In [ ]:
def reciprocal_rank(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> float:
    for rank, cid in enumerate(retrieved_ids[:k], start=1):
        if cid in relevant_ids:
            return 1.0 / rank
    return 0.0

def evaluate_results(results: list[dict], method: str, k_values: list[int]) -> tuple[dict, pd.DataFrame]:
    results_by_query = defaultdict(list)
    for row in results:
        results_by_query[row['query_id']].append(row)
    per_question = []
    for query in queries:
        qid = query['query_id']
        relevant_ids = qrels_by_query.get(qid, set())
        retrieved = sorted(results_by_query.get(qid, []), key=lambda r: r['rank'])
        retrieved_ids = [r['chunk_id'] for r in retrieved]
        row = {
            'method': method,
            'query_id': qid,
            'question': query['question'],
            'answer': query.get('answer'),
            'num_relevant_chunks': len(relevant_ids),
        }
        for k in k_values:
            top_ids = retrieved_ids[:k]
            hits = [cid for cid in top_ids if cid in relevant_ids]
            row[f'precision@{k}'] = len(hits) / k
            row[f'recall@{k}'] = len(set(hits)) / len(relevant_ids) if relevant_ids else math.nan
            row[f'hit@{k}'] = 1.0 if hits else 0.0
            row[f'mrr@{k}'] = reciprocal_rank(retrieved_ids, relevant_ids, k)
        per_question.append(row)
    per_df = pd.DataFrame(per_question)
    summary = {'method': method, 'queries': len(queries)}
    for k in k_values:
        for metric in ['precision', 'recall', 'hit', 'mrr']:
            summary[f'{metric}@{k}'] = float(per_df[f'{metric}@{k}'].mean())
    return summary, per_df

summaries = []
per_question_frames = []
for method, results in [
    ('dense_bge', dense_results),
    ('sparse_tfidf', sparse_results),
    (f'hybrid_bge_tfidf_alpha_{HYBRID_ALPHA:.2f}', hybrid_results),
]:
    summary, per_df = evaluate_results(results, method, K_VALUES)
    summaries.append(summary)
    per_question_frames.append(per_df)

summary_df = pd.DataFrame(summaries)
per_question_df = pd.concat(per_question_frames, ignore_index=True)
summary_df

In [ ]:
summary_df.to_json(OUT_DIR / 'metrics_summary.json', orient='records', indent=2, force_ascii=False)
per_question_df.to_json(OUT_DIR / 'metrics_by_question.jsonl', orient='records', lines=True, force_ascii=False)
all_results = dense_results + sparse_results + hybrid_results
write_jsonl(OUT_DIR / 'retrieval_results.jsonl', all_results)

display(summary_df[['method', 'precision@1', 'recall@1', 'hit@1', 'mrr@1', 'precision@3', 'recall@3', 'hit@3', 'mrr@3', 'precision@5', 'recall@5', 'hit@5', 'mrr@5', 'precision@10', 'recall@10', 'hit@10', 'mrr@10']])

## 6. Visual comparison

In [ ]:
plot_metrics = ['hit@10', 'recall@10', 'mrr@10', 'precision@10']
ax = summary_df.set_index('method')[plot_metrics].plot(kind='bar', figsize=(12, 5), title='Retrieval metrics @10')
ax.set_ylim(0, 1.0)
ax.set_ylabel('score')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 7. Error analysis

In [ ]:
hybrid_method = f'hybrid_bge_tfidf_alpha_{HYBRID_ALPHA:.2f}'
hybrid_errors = per_question_df[(per_question_df['method'] == hybrid_method) & (per_question_df['hit@10'] == 0)].copy()
print('Hybrid miss@10:', len(hybrid_errors))
hybrid_errors[['query_id', 'num_relevant_chunks', 'question', 'answer', 'hit@10', 'recall@10', 'mrr@10']].head(30)

In [ ]:
def show_retrieval_case(query_id: str, method: str = hybrid_method):
    print('QUERY')
    display(pd.DataFrame([q for q in queries if q['query_id'] == query_id]))
    print('RELEVANT CHUNKS')
    rel_ids = qrels_by_query[query_id]
    display(pd.DataFrame([
        {
            'chunk_id': cid,
            'page_start': page_range(chunks_by_id[cid])[0],
            'page_end': page_range(chunks_by_id[cid])[1],
            'text_preview': chunk_text(chunks_by_id[cid])[:500],
        }
        for cid in rel_ids
    ]))
    print('RETRIEVED TOP-K')
    result_rows = [r for r in all_results if r['query_id'] == query_id and r['method'] == method]
    display(pd.DataFrame(result_rows)[['rank', 'chunk_id', 'score', 'page_start', 'page_end', 'text_preview']])

if len(hybrid_errors):
    show_retrieval_case(hybrid_errors.iloc[0]['query_id'])
else:
    print('No hybrid miss@10 cases.')